In [ ]:
NUM_QUBITS = 3
Relation = [[1, 2]]


for i in range(1, 2* NUM_QUBITS - 1):
    if i % 2 == 1:
        Relation.append([i + 2, i + 3])
    else:
        Relation.append([i + 2, i + 2])


print(Relation)
print(Relation[-2][0])
# for i in range(1, NUM_QUBITS):
#     Mapping.append([2*i - 1, 2*i])
# Mapping.append([2*i + 1])
# Mapping.append([2*i + 2])

[[1, 2], [3, 4], [4, 4], [5, 6], [6, 6]]
5


In [11]:
import numpy as np
MATRIX_SIZE = 5
M = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20], [21, 22, 23, 24, 25]])

M[:MATRIX_SIZE-2, :MATRIX_SIZE-2]

array([[ 1,  2,  3],
       [ 6,  7,  8],
       [11, 12, 13]])

In [14]:
import numpy as np
from scipy.sparse import csr_matrix

dense = np.array([[0, 0, 3], [4, 0, 0]])
sparse = csr_matrix(dense)

print(sparse[0][0])


  (0, 2)	3


In [ ]:
def dd_sparse(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int):
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS)
    
    for op in operations:
        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
        
        # A = cp.eye(MATRIX_SIZE, dtype=cp.complex128)
        # B = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=cp.complex128)
        # C = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=cp.complex128)
        # D = cp.eye(MATRIX_SIZE, dtype=cp.complex128)
        
        # for node in Mapping[op.target]:
        #     A[node][node] = op.a
        #     B[node][node] = op.b
        #     C[node][node] = op.c
        #     D[node][node] = op.d
        
        A = cp.ones(MATRIX_SIZE, dtype=complex)
        A[Mapping[op.target]] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=complex)
        B[Mapping[op.target]] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=complex)
        C[Mapping[op.target]] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=complex)
        D[Mapping[op.target]] = op.d

        A = sparse.diags(A, offsets=0, format='csc')
        B = sparse.diags(B, offsets=0, format='csc')
        C = sparse.diags(C, offsets=0, format='csc')
        D = sparse.diags(D, offsets=0, format='csc')
    

        L_U = (L * A + R * B).tocsc()
        R_U = (L * C + R * D).tocsc()

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)
        
        elif isinstance(op, ControlledGate):
            
            # R = sparse.bmat([[R, None],
            #   [None, R_U]], format='csc')
            MATRIX_SIZE_OLD = MATRIX_SIZE
            for control, ctrl_cnt in enumerate(reversed(op.control)):
                # print("Control, count, Old, Now: ", control, ctrl_cnt, MATRIX_SIZE_OLD, MATRIX_SIZE)
                # print("Control: ", Mapping[control])
                # for node in Mapping[control]:
                #     R = sparse.hstack([R[:, :node], R[:, node + MATRIX_SIZE], R[:, node+1:MATRIX_SIZE*2]]).tocsc()
                

                # print("R: ", R)
                # print("R_U: ", R_U)

                keep = cp.zeros(MATRIX_SIZE_OLD, dtype=int)
                keep_complement = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                keep[Mapping[control]] = 1
                keep_complement[Mapping[control]] = 0
                D = sparse.diags(keep, offsets=0, format='csc')
                D_complement = sparse.diags(keep_complement, offsets=0, format='csc')
                # print("R_U.shape: ", R_U.shape)
                # print("D_complement.shape: ", D_complement.shape)
                # print("D.shape: ", D.shape)
                R_masked = R * D_complement
                R_U_masked = R_U[:, :MATRIX_SIZE_OLD] * D

                # print("R shape: ", R.shape)
                # print("R_U.shape: ", R_U.shape)
                # print("R Masked: ", R_masked.shape)
                # print("R U Masked: ", R_U_masked.shape)

                L_U = sparse.bmat([[L, None],
                    [None, L_U]], format='csc')
                
                R_U = sparse.bmat([[R_masked, None],
                    [R_U_masked, R_U]], format='csc')
                
                MATRIX_SIZE += MATRIX_SIZE_OLD

            L, R = L_U, R_U
            # print("start traverse")
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            # print(L_B)
            # print(R_B)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)
            # print(LR)

            traverse = sparse.csc_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)
            # print(traverse)
            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())
            # traverse = {0}
            # stack = {0}
            # while stack:
            #     now = stack.pop()
            #     # print("MATRIX_SIZE: ", MATRIX_SIZE)
            #     for i in range(MATRIX_SIZE*2):
            #         if i in traverse:
            #             continue
            #         if not L[i, now] == 0:
            #             traverse.add(i)
            #             stack.add(i)
            #         if not R[i, now] == 0:
            #             traverse.add(i)
            #             stack.add(i)
                    
            #     # print("Traverse: ", traverse)
            #     # print("stack: ", stack)

            #         # print("L: ", L)
            #         # print("R: ", R)
            #         # print("Mapping: ", Mapping)
            #     # print("Traverse: ", traverse)

            # traverse.sort()
            

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            Mapping_U = [
                sorted(
                    [n + MATRIX_SIZE for n in mapping if n + MATRIX_SIZE in traverse] +
                    [n for n in mapping if n in traverse]
                )
                for mapping in Mapping
            ]

            # print("Mapping: ", Mapping)
            # print("Mapping_U: ", Mapping_U)
            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            Mapping_U = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U: ", Mapping_U)
            Mapping = Mapping_U
            # print("end mapping, time spent:", time.time() - mapping_middle)  

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            # print("L: ", L)
            # print("R: ", R)
            
            # for t in not_traverse:
                # print("t: ", t)
                # print(L[:t, :t], L[:t, t+1:])
                # print(L[t+1:, :t], L[t+1:, t+1:])

                # print("L.shape: ", L.shape)
                # print("R.shape: ", R.shape)

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csc_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.complex128)
            
            # print("S Shape", S.shape)

            L = S * L * S.T
            R = S * R * S.T

                # L_T = sparse.hstack([L[:t, :t], L[:t, t+1:]])
                # L_B = sparse.hstack([L[t+1:, :t], L[t+1:, t+1:]])
                # L = sparse.vstack([L_T, L_B], format='csc')
                # R_T = sparse.hstack([R[:t, :t], R[:t, t+1:]])
                # R_B = sparse.hstack([R[t+1:, :t], R[t+1:, t+1:]])
                # R = sparse.vstack([R_T, R_B], format='csc')
                # print("L: ", L)
                # print("R: ", R)

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

            
            # print(L)
            # print(R)

            
        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")

    # print("MATRIX_SIZE: ", MATRIX_SIZE)
    # print("L: ", L)
    # print("R: ", R)
    # print("Mapping: ", Mapping)
    
    return L, R, Mapping